# Supply-chain research training on Kaggle (T4 x2 free tier)

Runbook for the cost-and-carbon NSGA-II + LSTM components on the Kaggle Notebooks free tier (NVIDIA T4 x2, 16 GB VRAM each, 9 hour wall-clock cap).

Reference docs:
* `cloud_training/README_CLOUD_SETUP.md` for the platform comparison.
* `cloud_training/TRAINING_GUIDE.md` for the ordered runbook across NSGA-II x 10 seeds, LSTM, PPO 1M-step.

All checkpoints are written to `/kaggle/working/checkpoints` so the same notebook can be re-run with `--resume <path>` to pick up after the 9 h cap.

In [ ]:
# Cell (b): GPU sanity check
!nvidia-smi

In [ ]:
# Cell (c): clone the repo and install pinned requirements
import os
REPO_URL = os.environ.get('REPO_URL', 'https://github.com/example/Supply-chain.git')
REPO_REF = os.environ.get('REPO_REF', 'main')
if not os.path.isdir('/kaggle/working/Supply-chain'):
    !git clone --depth 1 --branch {REPO_REF} {REPO_URL} /kaggle/working/Supply-chain
%cd /kaggle/working/Supply-chain
!pip install --quiet -r supply_chain_research/requirements.txt

In [ ]:
# Cell (d): mount the Kaggle dataset placeholder and copy the
# preprocessed .npy / .parquet artefacts the pipeline expects.
# Replace 'your-username/supply-chain-data' with the actual
# Kaggle dataset slug attached to this notebook.
import shutil
from pathlib import Path
DATASET_DIR = Path('/kaggle/input/supply-chain-data')
TARGET_DIR = Path('/kaggle/working/Supply-chain/data/processed')
TARGET_DIR.mkdir(parents=True, exist_ok=True)
if DATASET_DIR.exists():
    for src in DATASET_DIR.glob('*.npy'):
        shutil.copy2(src, TARGET_DIR / src.name)
    for src in DATASET_DIR.glob('*.parquet'):
        shutil.copy2(src, TARGET_DIR / src.name)
    print('Copied:', sorted(p.name for p in TARGET_DIR.iterdir()))
else:
    print('No Kaggle dataset attached; the runner will fall back '
          'to the synthetic generators in supply_chain_research.')
Path('/kaggle/working/checkpoints').mkdir(parents=True, exist_ok=True)

In [ ]:
# Cell (e): NSGA-II x 10 seeds (resumable from /kaggle/working)
!python cloud_training/local_training_runner.py --component nsga2 --seeds 10 --resume /kaggle/working/checkpoints/nsga2.pkl

In [ ]:
# Cell (f): LSTM demand forecaster
!python cloud_training/local_training_runner.py --component lstm --resume /kaggle/working/checkpoints/lstm.pt

In [ ]:
# Cell (g): summarise the artefacts produced this session
import json
from pathlib import Path
RESULTS = Path('/kaggle/working/Supply-chain/data/results')
for path in sorted(RESULTS.glob('*')):
    print(f'{path.name:40s} {path.stat().st_size:>12} bytes')
metrics = RESULTS / 'lstm_metrics.json'
if metrics.exists():
    print('\nLSTM metrics:')
    print(json.dumps(json.loads(metrics.read_text()), indent=2))